In [43]:
import pandas as pd
import ast
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file in project root
project_root = Path(__file__).parent.parent.parent if '__file__' in locals() else Path.cwd().parent.parent
dotenv_path = project_root / '.env'
load_dotenv(dotenv_path=dotenv_path) 

PAPER_PATH = Path(os.getenv('PAPER_PATH'))
print(PAPER_PATH)


/share/pierson/matt/UAIR/papers/uair_brief


In [44]:
DECOMPOSED_ARTICLES_PATH = "/share/pierson/matt/UAIR/outputs/decompose/oct14_1k_decompose_results.parquet"
decomposed_articles = pd.read_parquet(DECOMPOSED_ARTICLES_PATH)

In [45]:
decomposed_articles.columns

Index(['_progress_row', 'article_id', 'article_path', 'article_text',
       'batch_uuid', 'chunk_text', 'classification_mode', 'country',
       'embeddings', 'generated_text', 'generated_tokens', 'is_relevant',
       'keyword_match_count', 'latency_s', 'llm_output', 'matched_keywords',
       'messages', 'metrics', 'num_generated_tokens', 'num_input_tokens',
       'params', 'prompt', 'prompt_token_ids', 'relevance_answer',
       'relevant_keyword', 'request_id', 'time_taken_llm',
       'token_usage_output', 'token_usage_prompt', 'token_usage_total',
       'ts_start', 'year', 'llm_json', 'deployment_domain',
       'deployment_purpose', 'deployment_capability',
       'identity_of_ai_deployer', 'identity_of_ai_subject',
       'identity_of_ai_developer', 'location_of_ai_deployer',
       'location_of_ai_subject', 'date_and_time_of_event', 'deployment_space',
       'list_of_harms_that_occurred', 'list_of_risks_that_occurred',
       'list_of_benefits_that_occurred', 'missing'],
 

In [46]:
decomposed_articles['deployment_domain'].value_counts()

deployment_domain
Public and private transportation                                 129
Arts and Entertainment                                             86
Transport and Logistics                                            67
Education and vocational training                                  53
Smart home                                                         45
Finance and Investment                                             45
Energy                                                             44
Health and Healthcare                                              44
Media and Communication                                            37
Military and Defense                                               35
Marketing and Advertising                                          35
International Law Enforcement and Cooperation                      31
Government Services and Administration                             31
Urban Planning                                                     30
Ag

In [47]:
def flatten_string_lists(series):
    """Convert string representations of lists to actual lists, then flatten into one list"""
    all_items = []
    for item in series:
        # Convert string representation to actual list
        if isinstance(item, str):
            parsed = ast.literal_eval(item)
        else:
            parsed = item
        # Extend the all_items list with the parsed list
        if isinstance(parsed, list):
            all_items.extend(parsed)
    
    # Filter out empty elements (empty strings, None, whitespace-only strings)
    all_items = [item for item in all_items if item and str(item).strip()]
    
    return all_items

decomposed_articles_by_domain = decomposed_articles.groupby('deployment_domain')
# for each domain, aggregate via concatenation to get ALL risks, harms, and benefits mentioned in the group. each row is a list, so we need to flatten the lists into one list per domain
decomposed_articles_by_domain = decomposed_articles_by_domain.agg(
    num_articles=('article_id', 'nunique'),
    risks=('list_of_risks_that_occurred', flatten_string_lists),  
    harms=('list_of_harms_that_occurred', flatten_string_lists),
    benefits=('list_of_benefits_that_occurred', flatten_string_lists)

)

# drop empty lists and reset index 
decomposed_articles_by_domain = decomposed_articles_by_domain.dropna()
decomposed_articles_by_domain = decomposed_articles_by_domain.reset_index()

decomposed_articles_by_domain.sort_values(by='num_articles', ascending=False, inplace=True)







In [48]:
decomposed_articles_by_domain.head(n=10)

,deployment_domain,num_articles,risks,harms,benefits
32,Public and private transportation,129,[Potential widespread transmission of the coro...,[Travelers were exposed to health risks due to...,[Improved global health surveillance through e...
3,Arts and Entertainment,86,[Potential psychological harm to viewers due t...,[Individuals were exposed to traumatic memorie...,[Enhanced emotional depth in storytelling thro...
40,Transport and Logistics,67,[Potential endangerment of the driver due to m...,[The driver was injured due to a crash caused ...,[Expansion of self-driving test fleet to 150 r...
9,Education and vocational training,53,[Potential endangerment of public safety and p...,[A volunteer at Shenzhen Bao'an International ...,[Increased public awareness of ethical standar...
16,Finance and Investment,45,[Potential financial misrepresentation due to ...,[Auditors were forced to resign due to lack of...,[Real-time resolution of previously intractabl...
36,Smart home,45,[Potential privacy invasion due to unintended ...,[Apple users were surveilled without consent d...,[Improved selfie quality in low-light conditio...
20,Health and Healthcare,44,[Potential endangerment of patients due to AI ...,[Patients were misdiagnosed due to AI system f...,[Improved diagnostic speed for radiologists by...
11,Energy,44,[Potential reduction in renewable energy outpu...,[Solar panel efficiency was reduced due to ina...,[Enhanced learning outcomes in electrical safe...
29,Media and Communication,37,[Potential misuse of the news payment framewor...,[Legitimacy of public interest journalism was ...,[New report on AI in audiovisual creation rele...
31,Military and Defense,35,[Potential endangerment of civilians due to un...,[Residents of Belgorod Oblast lost electricity...,[Ukrainian military operations successfully di...


In [ ]:
# make a latex table of the top 10 domains, showing number of articles, and the risks, harms, and benefits 

# Configuration parameters
N_ITEMS_PER_CATEGORY = 6  # Number of risks/harms/benefits to show per domain
N_TOP_DOMAINS = 10  # Number of top domains to include

top_10_domains = decomposed_articles_by_domain.head(n=N_TOP_DOMAINS).copy()

def format_list_for_latex(items, max_items=None):
    """Format a list of items as a LaTeX itemize environment"""
    # Filter out empty elements (empty strings, None, whitespace-only strings)
    if items:
        items = [item for item in items if item and str(item).strip()]
    
    if not items or len(items) == 0:
        return "None reported"
    
    # Limit items if specified
    total_items = len(items)
    items_to_show = items[:max_items] if max_items else items
    is_truncated = max_items and total_items > max_items
    
    # Escape special LaTeX characters
    def escape_latex(text):
        text = str(text)
        # First, unescape any existing LaTeX escapes to avoid double-escaping
        text = text.replace(r'\$', '$').replace(r'\_', '_').replace(r'\&', '&').replace(r'\%', '%').replace(r'\#', '#')
        
        replacements = {
            '&': r'\&',
            '%': r'\%',
            '$': r'\$',
            '#': r'\#',
            '_': r'\_',
            '{': r'\{',
            '}': r'\}',
            '~': r'\textasciitilde{}',
            '^': r'\^{}',
        }
        for old, new in replacements.items():
            text = text.replace(old, new)
        return text
    
    # Create itemize list with smaller font
    escaped_items = [escape_latex(item) for item in items_to_show]
    itemized = r'\begin{itemize}[leftmargin=*, nosep, before=\vspace{-0.5\baselineskip}, after=\vspace{-0.5\baselineskip}] ' + '\n'
    itemized += r'\footnotesize' + '\n'
    for item in escaped_items:
        itemized += r'\item ' + item + '\n'
    
    # Add truncation note if applicable
    if is_truncated:
        itemized += r'\item \textit{[...and ' + str(total_items - max_items) + r' more]}' + '\n'
    
    itemized += r'\end{itemize}'
    
    return itemized

# Build custom LaTeX table manually for better formatting
def build_custom_latex_table(domains_df, n_items):
    """Build a custom LaTeX table with domain headers spanning all columns"""
    
    # Start the longtable environment
    latex = r'\begin{longtable}{p{5.2cm}|p{5.2cm}|p{5.2cm}}' + '\n'
    latex += r'\caption{Top ' + str(len(domains_df)) + r' AI Deployment Domains by Article Count with Reported Risks, Harms, and Benefits}' + '\n'
    latex += r'\label{tab:top_domains_detailed}\\' + '\n'
    
    # First head - empty, as we'll add domain-specific headers
    latex += r'\endfirsthead' + '\n\n'
    
    # Continued header for subsequent pages - also empty
    latex += r'\endhead' + '\n\n'
    
    # Footer for non-final pages - empty
    latex += r'\endfoot' + '\n\n'
    
    # Final footer
    latex += r'\endlastfoot' + '\n\n'
    
    # Add each domain
    for idx, row in domains_df.iterrows():
        domain = row['deployment_domain']
        n_articles = row['num_articles']
        risks = row['risks']
        harms = row['harms']
        benefits = row['benefits']
        
        # Domain header spanning all columns
        latex += r'\multicolumn{3}{l}{\textbf{\large ' + escape_latex(domain) + r'} \textit{(N=' + str(n_articles) + r' articles)}} \\' + '\n'
        latex += r'\midrule' + '\n'
        
        # Column headers for this domain
        latex += r'\textbf{Risks} & \textbf{Harms} & \textbf{Benefits} \\' + '\n'
        latex += r'\midrule' + '\n'
        
        # Format the three columns
        risks_text = format_list_for_latex(risks, max_items=n_items)
        harms_text = format_list_for_latex(harms, max_items=n_items)
        benefits_text = format_list_for_latex(benefits, max_items=n_items)
        
        # Add content row
        latex += risks_text + ' & ' + harms_text + ' & ' + benefits_text + r' \\' + '\n'
        latex += r'\pagebreak' + '\n'
    
    # End the table
    latex += r'\end{longtable}' + '\n'
    
    return latex

def escape_latex(text):
    """Escape special LaTeX characters"""
    text = str(text)
    # First, unescape any existing LaTeX escapes to avoid double-escaping
    text = text.replace(r'\$', '$').replace(r'\_', '_').replace(r'\&', '&').replace(r'\%', '%').replace(r'\#', '#')
    
    replacements = {
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

# Generate the custom table
latex_table = build_custom_latex_table(top_10_domains, N_ITEMS_PER_CATEGORY)

# Print the LaTeX table code
print("% Required packages in your main document preamble:")
print("% \\usepackage{longtable}")
print("% \\usepackage{booktabs}")
print("% \\usepackage{enumitem}")
print("% \\usepackage{array}")
print()
print("% Table code (ready for \\input):")
print()
print(latex_table)

# write to .tex file in papers directory / tables 
with open(PAPER_PATH / 'tables' / 'top_domains_detailed.tex', 'w') as f:
    f.write(latex_table)



% Required packages in your main document preamble:
% \usepackage{longtable}
% \usepackage{booktabs}
% \usepackage{enumitem}
% \usepackage{array}

% Table code (ready for \input):

\begin{longtable}{p{4.5cm}|p{4.5cm}|p{4.5cm}}
\caption{Top 10 AI Deployment Domains by Article Count with Reported Risks, Harms, and Benefits}
\label{tab:top_domains_detailed}\\
\endfirsthead

\endhead

\endfoot

\endlastfoot

\multicolumn{3}{l}{\textbf{\large Public and private transportation} \textit{(N=129 articles)}} \\
\midrule
\textbf{Risks} & \textbf{Harms} & \textbf{Benefits} \\
\midrule
\begin{itemize}[leftmargin=*, nosep, before=\vspace{-0.5\baselineskip}, after=\vspace{-0.5\baselineskip}] 
\footnotesize
\item Potential widespread transmission of the coronavirus in the United States due to delayed public health response despite real-time monitoring of global travel patterns.
\item Potential breach of user privacy due to Apple and Google's joint contact-tracing technology collecting Bluetooth proxim